<a href="https://colab.research.google.com/github/dnhshl/cc-ai/blob/main/llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mission Planning mit Generativer KI (LLM)
### KI in der Robotik

Nutze **Large Language Models (LLMs)**, um natürliche Sprache in strukturierte Befehle zu übersetzen.

26.01.2026 DN

In [ ]:
# 1. Installation und Setup
!pip install -q -U google-generativeai

import google.generativeai as genai
import json
import os
from google.colab import userdata # Bibliothek für den sicheren Zugriff

# API Key sicher aus den Colab Secrets laden
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    genai.configure(api_key=GOOGLE_API_KEY)
    print("ERFOLG: API Key wurde sicher aus den Secrets geladen.")
except Exception as e:
    print("FEHLER: Konnte den API Key 'GOOGLE_API_KEY' nicht finden.")
    print("Bitte links auf das Schlüssel-Symbol klicken und Secret anlegen!")

## 2. Der System-Prompt (Die "Persönlichkeit" des Roboters)
Wir müssen dem LLM erklären, welche Fähigkeiten der Roboter physisch besitzt. Dies nennt man **Grounding**.
Der Roboter darf nicht halluzinieren (z.B. "Ich fliege zum Mond"), sondern darf nur definierte Funktionen nutzen:
* `move_to(object)`
* `grab()`
* `release()`
* `find_object(name)`

In [ ]:
# 2. System-Prompt definieren (Das "Grounding")
system_instruction = """
Du bist das Gehirn eines Service-Roboters. Übersetze Befehle in eine JSON-Liste.
Fähigkeiten:
1. move_to(target): Fährt zu Objekt/Ort (z.B. 'bottle', 'trash_bin', 'apple', 'user').
2. grab(): Greift das Objekt vor sich.
3. release(): Lässt das Objekt los.
4. find_object(name): Sucht ein Objekt im Raum.

Regeln:
- Antworte NUR mit validem JSON.
- Format: [{"action": "move_to", "target": "apple"}, {"action": "grab"}]
"""

# Modell initialisieren
model = genai.GenerativeModel(
    model_name='gemini-3-flash-preview',
    system_instruction=system_instruction,
    generation_config={"response_mime_type": "application/json"}
)



In [ ]:
# --- Aufruf der LLM mit einem Kommando ---

def generate_plan(command_string):
    """Sendet den Befehl an das LLM und gibt den geparsten Plan zurück."""
    print(f"Denke nach über: '{command_string}'...")
    try:
        response = model.generate_content(command_string)
        plan = json.loads(response.text)
        print(f"JSON Plan:\n{plan}")
        return plan
    except Exception as e:
        print(f"Fehler bei der Plan-Erstellung: {e}")
        return []



In [ ]:
# --- Ausgabe des generierten Plans ---

def print_plan(plan):
    """Simuliert die Ausführung des Plans für den Roboter."""
    print("\n--- Mission-Plan ---")
    if not plan:
        print("Plan ist leer oder ungültig.")
        return

    for i, step in enumerate(plan, 1):
        action = step.get('action')
        target = step.get('target', '')

        # Simulation der Hardware-Calls
        if action == 'move_to':
            print(f"{i}. 📍 Fahre zu: [{target}] (Navigations-Stack aktiv)")
        elif action == 'grab':
            print(f"{i}. ✊ Greife Objekt (YOLO bestätigt Position -> SAM generiert Maske -> Greifer zu)")
        elif action == 'release':
            print(f"{i}. ✋ Lasse Objekt los")
        elif action == 'find_object':
            print(f"{i}. 🔎 Suche nach: [{target}] (Objekterkennung läuft)")
        else:
            print(f"{i}. ❓ Unbekannte Aktion: {action}")
    print("-----------------------\n")


## 3. Der Test: Vom Text zur Aktion
Wir geben nun einen komplexen, verschachtelten Befehl ein. Beobachten Sie, wie das LLM die Logik (Reihenfolge, Kontext) versteht.

In [ ]:
# Der Befehl des Benutzers
user_command = "Die rote Flasche ist leer. Wirf sie in den Mülleimer und bring mir danach den Apfel."
#user_command = "Ich bin durstig."

print(f"User Command: '{user_command}'\n")

In [ ]:


# Anfrage an die KI
plan = generate_plan(user_command)

# Ergebnis parsen
print_plan(plan)


Das Modell hat verstanden:
1.  **Kontext:** "Wirf sie weg" bezieht sich auf die "rote Flasche".
2.  **Ziel:** "Mülleimer" ist das Ziel für das Wegwerfen, obwohl es nicht explizit als "Fahre zum Mülleimer" gesagt wurde.
3.  **Sequenz:** "Danach" bedeutet, erst die erste Aufgabe beenden (release), dann die neue beginnen (move_to apple).

Das ist **Semantische Robotik**.

## Multimodales Reasoning

Verarbeite Bildinformation und Anweisungen

In [ ]:
# Hilfsfunktion, um Bild zu laden

def load_image_from_url(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content))
    return img

In [ ]:
# Lade Bild
import requests
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

image_url = "https://images.pexels.com/photos/7722857/pexels-photo-7722857.jpeg"

print("Lade Bild...")
img = load_image_from_url(image_url)

# Bild anzeigen (damit die Studenten sehen, worum es geht)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# --- Multimodaler Aufruf der LLM  ---

def multimodal_plan(command_string, img):
    """Sendet den Befehl an das LLM und gibt den geparsten Plan zurück."""
    print(f"Denke nach über: '{command_string}'...")
    try:
        response = model.generate_content([command_string, img])
        plan = json.loads(response.text)
        print(f"JSON Plan:\n{plan}")
        return plan
    except Exception as e:
        print(f"Fehler bei der Plan-Erstellung: {e}")
        return []


In [ ]:
# Der Befehl des Benutzers
user_command = "Analysiere das Bild. Erkenne die Buchstaben. Sortiere sie nach dem Alphabet"

print(f"User Command: '{user_command}'\n")

In [ ]:

# Anfrage an die KI
plan = multimodal_plan(user_command, img)

# Ergebnis parsen
print_plan(plan)